# Groundhog Day Predictions

**Tabular Multi-Class Classification** — Predict Punxsutawney Phil's shadow prediction.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Dataset: Groundhog Day (132 rows × 10 columns)  
Target: `Punxsutawney Phil` (shadow prediction)

## 2 · Project Overview

We predict whether **Punxsutawney Phil** (the famous groundhog) will see his shadow on Groundhog Day, based on historical February and March temperature data.

This is a fun, educational dataset with extreme small-data challenges (132 rows, many missing values).

## 3 · Learning Objectives

1. Handle a very small dataset (132 rows) with missing values.
2. Engineer features from temperature data.
3. Build classifiers for a quirky real-world problem.
4. Understand the limits of ML on tiny, noisy datasets.
5. Compare model performance vs random guessing.

## 4 · Problem Statement

Given February and March average temperatures for different US regions, predict Punxsutawney Phil's prediction: **Full Shadow**, **No Shadow**, **Partial Shadow**, or **No Record**.

## 5 · Why This Project Matters

- Fun, engaging dataset for ML education.
- Tests ML on tiny, noisy, real-world data.
- Demonstrates that ML can't always find signal.
- Teaching moment: not every problem benefits from complex models.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 132 (1886–2017) |
| **Columns** | 10 |
| **Features** | Year, 8 temperature columns |
| **Target** | `Punxsutawney Phil` |
| **Classes** | Full Shadow, No Shadow, Partial Shadow, No Record |
| **Missing values** | Many (early years have no temperature data) |
| **Source** | Local dataset.csv |

## 7 · Dataset Source and License Notes

- **Source**: Stormfax Weather Almanac / Kaggle.
- **License**: Public / educational use.
- **Note**: Combines groundhog predictions (since 1886) with NOAA temperature data.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, re, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "Punxsutawney Phil"
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "dataset.csv")
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Data path: {DATA_PATH}")
print(f"Save dir : {SAVE_DIR}")

## 11 · Dataset Download or Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nShape: {df.shape}")

## 13 · Exploratory Data Analysis

In [ ]:
# Temperature features over time
temp_cols = [c for c in df.columns if "Temperature" in c]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(temp_cols[:4]):
    ax = axes[i // 2, i % 2]
    valid = df[["Year", col]].dropna()
    ax.scatter(valid["Year"], valid[col], alpha=0.6, s=20, color="steelblue")
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("Year")
    ax.set_ylabel("Temperature (°F)")
plt.suptitle("Temperature Trends Over Time", fontsize=14)
plt.tight_layout()
plt.show()

## 14 · Target Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title(f"Distribution: {TARGET}")
ax.set_ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

for val in df[TARGET].value_counts().index:
    n = (df[TARGET] == val).sum()
    print(f"  {val}: {n} ({100*n/len(df):.1f}%)")

## 15 · Train/Validation/Test Split Strategy

Drop rows with missing target or missing ALL temperature features. Stratified 80/20 split.

In [ ]:
# Clean up: drop No Record and missing values
temp_cols = [c for c in df.columns if "Temperature" in c]

# Remove No Record
df_clean = df[df[TARGET] != "No Record"].copy()
print(f"After removing 'No Record': {df_clean.shape}")

# Drop rows where ALL temperature features are missing
df_clean = df_clean.dropna(subset=temp_cols, how="all")
print(f"After dropping rows with no temp data: {df_clean.shape}")

# Fill remaining missing with median
for col in temp_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Features: Year + temperature columns
X = df_clean[["Year"] + temp_cols]
y_raw = df_clean[TARGET].values

# Encode target
le = LabelEncoder()
y = le.fit_transform(y_raw)
n_classes = len(le.classes_)
print(f"Classes ({n_classes}): {list(le.classes_)}")

# Sanitize column names
X.columns = [re.sub(r"[^A-Za-z0-9_]", "_", c) for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Removed**: 'No Record' rows (no prediction made).
- **Missing temperatures**: Filled with column median.
- **No scaling**: Tree models handle raw values.
- **Dropped**: Rows with ALL temperatures missing.

## 17 · Feature Engineering

Using Year and 8 temperature columns directly. The dataset is too small for complex feature engineering to help.

## 18 · Baseline Model

In [ ]:
baseline = DecisionTreeClassifier(max_depth=3, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="weighted")

print("=" * 50)
print("BASELINE: Decision Tree (max_depth=3)")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print(f"\n{classification_report(y_test, y_pred_bl, target_names=le.classes_)}")

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                     metric="f1", verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    acc_flaml = accuracy_score(y_test, y_pred_flaml)
    f1_flaml = f1_score(y_test, y_pred_flaml, average="weighted")
    print(f"FLAML best: {flaml_automl.best_estimator}")
    print(f"Accuracy: {acc_flaml:.4f}, F1: {f1_flaml:.4f}")
except Exception as e:
    print(f"FLAML failed: {e}")
    print("Using baseline predictions as FLAML fallback.")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    cb = CatBoostClassifier(iterations=200, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    lg = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                            device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
    lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
           callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                              device="cuda", tree_method="hist", verbosity=0,
                              eval_metric="mlogloss", n_jobs=-1, random_state=SEED)
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost: {e}")
gpu_cleanup()

## 22 · Top 3 Model Selection

In [ ]:
# Add baseline and FLAML
results["Baseline"] = y_pred_bl
results["FLAML"] = y_pred_flaml

model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], colorbar=False)
    f1 = f1_score(y_test, yp, average='weighted')
    axes[i].set_title(f"{name}\nF1={f1:.4f}")

plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    y_t = y_test if isinstance(y_test, np.ndarray) else y_test.values
    print(f"\n  {name}:")
    print(f"    Accuracy: {accuracy_score(y_t, yp):.4f}")
    print(f"    F1      : {f1_score(y_t, yp, average='weighted'):.4f}")
    print(f"    Precision: {precision_score(y_t, yp, average='weighted'):.4f}")
    print(f"    Recall  : {recall_score(y_t, yp, average='weighted'):.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]

y_t = y_test if isinstance(y_test, np.ndarray) else y_test.values
misclassed = (y_t != best_pred)
n_wrong = misclassed.sum()
print(f"Best model: {best_name}")
print(f"Misclassified: {n_wrong} / {len(y_t)} ({100*n_wrong/len(y_t):.1f}%)")

if n_wrong > 0:
    print(f"\nClassification Report ({best_name}):")
    print(classification_report(y_t, best_pred))

## 25 · Interpretation and Business Insight

**Key findings:**
- **Phil almost always sees his shadow** (~80%), making majority-class prediction hard to beat.
- **Temperature features** have weak predictive power for Phil's prediction.
- **Phil's prediction is not based on actual weather** — it's a tradition, not meteorology.

**Takeaway:** This is a great example of a dataset where ML has limited value — the "signal" is noise because the groundhog's prediction is not data-driven.

## 26 · Limitations

1. Only ~100 usable rows after cleaning.
2. Phil's prediction is a tradition, not a scientific measurement.
3. Temperature data has many missing values in early years.
4. Class imbalance (Full Shadow ~80%).
5. Temporal confound — climate change trends vs prediction patterns.

## 27 · How to Improve This Project

1. Frame as binary: Shadow vs No Shadow.
2. Add weather features for Gobbler's Knob specifically.
3. Add historical sunrise/cloud cover data.
4. Try time-series approach (autocorrelation of predictions).
5. Test if Phil's accuracy vs actual spring arrival is better than random.

## 28 · Production Considerations

- This is a fun educational project — not for production deployment.
- Could be a novelty prediction widget for news outlets.
- Demonstrates importance of domain understanding.

## 29 · Common Mistakes

1. Expecting ML to find signal in non-scientific data.
2. Not removing 'No Record' entries.
3. Using accuracy on 80/20 imbalanced class.
4. Overfitting on 100 rows.
5. Not acknowledging that Phil's prediction is arbitrary.

## 30 · Mini Challenge / Exercises

1. Merge to binary (Shadow/No Shadow) — does accuracy improve?
2. Plot Phil's accuracy vs actual spring temperatures.
3. Use only Year as feature — does it predict as well?
4. Try a naive baseline: always predict 'Full Shadow'.
5. Compare Phil's forecast accuracy to a coin flip.

## 31 · Final Summary / Key Takeaways

1. **Not every dataset has learnable signal** — this is a key ML lesson.
2. **Phil always sees shadow** is hard to beat with any model.
3. **Small datasets** with weak features frustrate all algorithms equally.
4. **Domain knowledge** is essential — Phil's prediction is a tradition, not meteorology.
5. **Baseline comparison** reveals when ML adds no value.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    y_t = y_test if isinstance(y_test, np.ndarray) else y_test.values
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_t, yp)), 4),
        "f1": round(float(f1_score(y_t, yp, average='weighted')), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))